## Choosing the History That a Forecaster Reads

A forecaster of tomorrow's load has to be given a part of the past to read. Three choices are involved: the length of the window of recent observations, the older values that are added to it, and the other inputs, such as the hour, the weekday or the temperature. The window and the lags are often set by convention, for example one week of history, and the inputs by trial on the test set, which lets the test period influence the model that is meant to predict it.

This post makes the three choices from the training data alone, with the functions of the open-source library twiga.[^twiga] The autocorrelation function sets the lookback window and proposes the lags. The lags, the calendar variables and the known weather inputs are then ranked by six measures of relevance, first within their own subgroup and then together, so that they compete on one scale. A column of random noise is placed among the candidates as a control, because a ranking that places noise high cannot be trusted. The method is applied to four series with different structure: the half-hourly demand of one London household, the hourly charging of electric vehicles (EV) across Los Angeles, the output of a photovoltaic (PV) array, and the net load of the site that the array feeds.

The post stops at the selection. It does not train a forecaster, because the claim made here is narrower than a claim about accuracy: the choices are made on training data only, they can be inspected, and they behave sensibly on a control. Chapter 2 of *Forecasting for Operations* develops the same method on synthetic series with a known answer,[^book] and the figures below follow its design. The code is shown in full and is folded by default. Every function that makes a selection is imported from twiga 1.1.8 and not rewritten.

[^twiga]: Version 1.1.8 from the Python Package Index, installed with the extra ml. Documentation: [twiga-docs](https://sambaiga.github.io/twiga-docs/).

[^book]: Volume 2 of *Machine Learning for Time Series*, read online at [the mbuga site](https://sambaiga.github.io/mbuga/operations/).

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from importlib.metadata import version
import re
from pathlib import Path
import tempfile
import urllib.request

from lets_plot import (
    aes, arrow, gggrid, element_blank, element_line, element_rect, element_text, facet_wrap, geom_abline, geom_band, geom_hline, geom_line,
    geom_point, geom_rect, geom_ribbon, geom_segment, geom_text, geom_tile, geom_vline, ggplot, ggsize, labs, scale_color_manual,
    scale_fill_gradient, scale_fill_manual, scale_size_manual, scale_x_continuous, scale_x_discrete, scale_y_continuous,
    scale_y_discrete, scale_y_reverse, theme,
)
import numpy as np
import pandas as pd
from twiga.core.data import AutoregressTransformer, add_fourier_features, select_top_features
from twiga.core.data.feature import add_temporal_feature
from twiga.core.data.selection import diversity_weights
from twiga.core.stats.seasonality import get_acf_values
from twiga.utils.data_loading import get_best_autoregessive_features

from IPython.display import HTML, display

pd.set_option("display.max_colwidth", None)

# Colours by role: subject blue, highlight orange, reference slate, context grey.
BLUE, ORANGE, SLATE, INK = "#0369A1", "#E26029", "#64748B", "#1E293B"
GREY, GREY_DARK, FONT = "#CED4DA", "#6B7280", "'Libre Franklin', 'Segoe UI', Roboto, Helvetica, Arial, sans-serif"


def tint(colour: str, amount: float = 0.85) -> str:
    """A colour mixed with white, for bands and fills."""
    rgb = np.array([int(colour[i : i + 2], 16) for i in (1, 3, 5)])
    return "#" + "".join(f"{int(v):02X}" for v in rgb + (255 - rgb) * amount)


def house() -> theme:
    """Thin dark axes on the left and bottom, no grid, no legend, Libre Franklin type."""
    return theme(
        legend_position="none", plot_background=element_rect(fill="white", color="white"), panel_background=element_blank(),
        panel_border=element_blank(), panel_grid=element_blank(), strip_background=element_blank(),
        strip_text=element_text(face="bold", size=12, hjust=0), axis_line=element_line(size=0.5, color=INK),
        axis_ticks=element_line(size=0.5, color=INK), axis_ticks_length=4,
        text=element_text(family=FONT, size=11, color=INK), plot_margin=[8, 18, 4, 4],
    )


def show(plot) -> None:
    """Draw a lets-plot figure as inline SVG, so that it scales with the page, takes the page fonts and needs no script."""
    svg = plot.to_svg()
    width, height = (float(v) for v in re.search(r'width="([\d.]+)" height="([\d.]+)"', svg).groups())
    svg = svg.replace(f'width="{width}" height="{height}"', f'viewBox="0 0 {width:g} {height:g}" width="100%"', 1)
    display(HTML(f'<div class="letsplot">{svg}</div>'))


def note(frame: pd.DataFrame, colour: str = INK, hjust: float = 0.0, size: float = 4.6):
    """Direct labels from a frame with columns x, y, label (and the facet column), with a white halo over the marks."""
    return geom_text(
        aes("x", "y", label="label"), data=frame, color=colour, hjust=hjust, size=size, family=FONT,
        halo_width=2, halo_color="white", inherit_aes=False,
    )


print("twiga", version("twiga"), "| lets-plot", version("lets-plot"))

## Four Series with Different Structure

The household is the half-hourly demand of one customer of the Low Carbon London trial, taken from the Monash collection on Hugging Face.[^lcl] The EV series sums the hourly energy charged at all the charging sites of Los Angeles over six months, from the CHARGED dataset.[^charged] The PV and net-load series come from one site in Plymouth, United Kingdom, at half-hourly resolution, with weather from the same area.[^plymouth] Net load is the demand of the site minus the output of the array, so the two series share their weather and differ in what they measure.

@fig-series shows one week of each series. The inputs that are known in advance, such as the irradiance, are treated as available over the forecast period, as a weather forecast would supply them. A real forecast carries the error of that weather forecast, which this post does not examine. The household and the Plymouth data are stored beside the notebook as small files. The EV series is downloaded when the notebook runs, because the dataset card states no licence for redistribution.

[^lcl]: Low Carbon London trial of UK Power Networks, through the monash_london_smart_meters configuration of autogluon/chronos_datasets (CC BY 4.0). One household, last 20,000 half-hours of its record.

[^charged]: yzzpz/Charged-data on Hugging Face, folder LOA. The card gives no licence.

[^plymouth]: Electricity demand data and solar generation data from Plymouth, UK, Zenodo record 5500457 (CC BY 4.0, under the data owner's open data licence).

In [ ]:
DATA = Path("data")
EV_URL = "https://huggingface.co/datasets/yzzpz/Charged-data/resolve/main/LOA/volume.csv"

household = pd.read_parquet(DATA / "household.parquet").rename(columns={"demand_kwh": "household"})
plymouth = pd.read_parquet(DATA / "plymouth.parquet")
plymouth["netload"] = plymouth["demand_mw"] - plymouth["pv_power_mw"]

ev_file = Path(tempfile.gettempdir()) / "charged-los-angeles-volume.csv"
if not ev_file.exists():
    urllib.request.urlretrieve(EV_URL, ev_file)
charging = pd.read_csv(ev_file, index_col=0, parse_dates=True).sum(axis=1)

SERIES = {
    "Household demand": {"data": household, "target": "household", "known": [], "samples": 48, "unit": "kWh per half-hour"},
    "EV charging": {"data": pd.DataFrame({"timestamp": charging.index, "ev": charging.to_numpy()}), "target": "ev", "known": [], "samples": 24, "unit": "kWh per hour"},
    "PV output": {
        "data": plymouth[["timestamp", "pv_power_mw", "irradiance_wm2", "panel_temp_c"]].rename(columns={"pv_power_mw": "pv"}),
        "target": "pv", "known": ["irradiance_wm2", "panel_temp_c"], "samples": 48, "unit": "MW",
    },
    "Net load": {"data": plymouth[["timestamp", "netload", "air_temp_c", "solar_wm2"]], "target": "netload", "known": ["air_temp_c", "solar_wm2"], "samples": 48, "unit": "MW"},
}
TAGS = {name: f"({chr(97 + i)}) {name}" for i, name in enumerate(SERIES)}  # panel tags, which also fix the panel order
for series in SERIES.values():
    series["data"] = series["data"].ffill()  # twiga's own loader forward-fills short gaps in the same way

for name, s in SERIES.items():
    d = s["data"]
    print(f"{name}: {len(d):,} rows, {d['timestamp'].min():%Y-%m-%d} to {d['timestamp'].max():%Y-%m-%d}, {s['samples']} steps a day, known inputs: {', '.join(s['known']) or 'none'}")

In [ ]:
#| label: fig-series
#| fig-cap: "One week of each series, from a Monday of its training window. The daily cycle is clear in the charging, the PV output and the net load, and weak in the household."
rows = []
for name, spec in SERIES.items():
    data, n = spec["data"], spec["samples"]
    start = int(len(data) * 0.4)
    start += (7 - data["timestamp"].iloc[start].dayofweek) % 7 * n  # begin on a Monday
    week = data.iloc[start : start + 7 * n]
    rows.append(pd.DataFrame({"series": f"{TAGS[name]} ({spec['unit']})", "day": np.arange(len(week)) / n, "value": week[spec["target"]].to_numpy()}))
weeks = pd.concat(rows)

show(
    ggplot(weeks, aes("day", "value"))
    + geom_line(color=BLUE, size=0.6)
    + facet_wrap("series", ncol=2, scales="free_y")
    + scale_x_continuous(breaks=list(range(8)))
    + labs(x="Days from Monday", y="")
    + house() + ggsize(720, 400)
)

## Lookback Window and Lags

A direct forecaster maps inputs that are observed at the forecast origin to the values of the next day. Two kinds of input come from the history of the target itself. The *lookback window* of length $W$ holds the most recent $W$ observations as one block that ends at the origin. A *lag* of $k$ steps attaches to every step of the window the value $y_{t-k}$, which reaches further back than the window without lengthening it. A lag is admissible when its value exists at the origin, so that every lag is at least one step. @fig-schematic shows the window, the horizon and three lags for a forecast one day ahead.

Both choices have a cost. A window that is too short hides the daily shape, and a window that is too long, or a lag that repeats the window, gives the model inputs that carry the same information more than once. That raises its variance without adding signal. The autocorrelation function (ACF) gives a principled place to start.[^acf] It is the correlation between the series and itself $k$ steps earlier,

$$
\rho_k = \frac{\operatorname{Cov}(y_t,\, y_{t-k})}{\operatorname{Var}(y_t)},
$$

and its peaks at multiples of a period show the periods at which the series repeats. The shortest significant peak sets the window, as the shortest span over which the dynamics repeat, and the longer peaks become candidates for lags. The window already covers the shortest period, so a lag of that length is left out. A value of $\rho_k$ is significant when it lies outside the band $\pm 1.96/\sqrt{T}$ of a series of length $T$ with no autocorrelation.

[^acf]: G. E. P. Box, G. M. Jenkins, G. C. Reinsel and G. M. Ljung, *Time Series Analysis: Forecasting and Control*, 5th ed., Wiley, 2015.

In [ ]:
#| label: fig-schematic
#| fig-cap: "A forecast one day ahead. The lookback window covers the last day before the origin, the horizon the next day, and each lag adds the value two, three or seven days earlier than each step of the window."
lags = (2, 3, 7)
boxes = pd.DataFrame(
    [(-1, 0, 0.0, 0.5, "Window"), (0, 1, 0.0, 0.5, "Horizon")]
    + [(-1 - d, -d, -0.95 * row, -0.95 * row + 0.5, "Lag") for row, d in enumerate(lags, start=1)],
    columns=["x0", "x1", "y0", "y1", "kind"],
)
arrows = pd.DataFrame({"x": -1.08, "xend": [-d + 0.06 for d in lags], "y": [-0.95 * row + 0.25 for row in range(1, 4)]})
texts = pd.DataFrame(
    [(-0.1, 0.78, "Lookback window: one day", 1.0), (0.1, 0.78, "Horizon: next day", 0.0), (0.0, 1.45, "Forecast origin", 0.5)]
    + [(0.25, -0.95 * row + 0.25, f"Lag of {d} days", 0.0) for row, d in enumerate(lags, start=1)],
    columns=["x", "y", "label", "hjust"],
)

show(
    ggplot()
    + geom_rect(aes(xmin="x0", xmax="x1", ymin="y0", ymax="y1", fill="kind", color="kind"), data=boxes, size=0.8)
    + scale_fill_manual(values={"Window": BLUE, "Horizon": "white", "Lag": tint(SLATE, 0.55)})
    + scale_color_manual(values={"Window": BLUE, "Horizon": SLATE, "Lag": tint(SLATE, 0.55)})
    + geom_segment(aes(x="x", xend="xend", y="y", yend="y"), data=arrows, color=SLATE, size=0.6, arrow=arrow(length=7, type="closed"))
    + geom_segment(aes(x="x", xend="x", y="y0", yend="y1"), data=pd.DataFrame({"x": [0.0], "y0": [-3.0], "y1": [1.0]}), color=INK, linetype="dashed", size=0.5, inherit_aes=False)
    + geom_text(aes("x", "y", label="label", hjust="hjust"), data=texts, family=FONT, size=4.6, color=INK, inherit_aes=False)
    + scale_x_continuous(breaks=list(range(-8, 1)), labels=[str(-b) for b in range(-8, 1)], limits=[-8.5, 4.2])
    + labs(x="Days before the origin", y="")
    + house() + theme(axis_line_y=element_blank(), axis_ticks_y=element_blank(), axis_text_y=element_blank()) + ggsize(720, 330)
)

The analysis runs on the first 70% of each series in time, so that nothing from the period to be forecast enters the choice. twiga's function for autoregressive features returns the window and builds the lags as calendar multiples of the period that it detects.

In [ ]:
TRAIN_SHARE = 0.7


def training(name: str, share: float = TRAIN_SHARE) -> pd.DataFrame:
    """The first ``share`` of a series in time: the only part that any selection below may read."""
    data = SERIES[name]["data"]
    return data.iloc[: int(len(data) * share)].copy()


def lookback_and_lags(name: str, share: float = TRAIN_SHARE) -> tuple[int, list[int]]:
    """Lookback window in steps and lag candidates in days, from twiga's autocorrelation analysis of the training window."""
    series = SERIES[name]
    candidates, lookback = get_best_autoregessive_features(
        training(name, share), series["target"], n_samples=series["samples"], top_k=50
    )
    return lookback, sorted({int(feature.rsplit("_", 1)[1]) // series["samples"] for feature in candidates})


hints = {name: lookback_and_lags(name) for name in SERIES}
for name, (lookback, days) in hints.items():
    print(f"{name}: {len(training(name)):,} training rows, lookback {lookback} steps ({lookback / SERIES[name]['samples']:.0f} day), lag candidates {days} days")

In [ ]:
#| label: fig-acf
#| fig-cap: "Autocorrelation of the training window over seven days. The blue band is the lookback window, the grey band the range of no autocorrelation, and the dots the lag candidates with their values."
frames = {"acf": [], "band": [], "window": [], "dots": []}
for name, spec in SERIES.items():
    n, target = spec["samples"], spec["target"]
    values = training(name)[target].to_numpy()
    acf, _ = get_acf_values(values, maxlag=7 * n + n // 2, seasonal_length=n)
    days = np.arange(1, len(acf) + 1) / n
    lookback, lags = hints[name]
    band = 1.96 / np.sqrt(len(values))
    frames["acf"].append(pd.DataFrame({"series": name, "lag": days, "acf": acf}))
    frames["band"].append(pd.DataFrame({"series": name, "lag": [0, 7.5], "lo": -band, "hi": band}))
    frames["window"].append(pd.DataFrame({"series": [name], "x0": [0.0], "x1": [lookback / n]}))
    frames["dots"].append(pd.DataFrame({"series": name, "x": lags, "y": [acf[lag * n - 1] for lag in lags], "label": [f"{lag} d: {acf[lag * n - 1]:.2f}" for lag in lags]}))
acf_df, band_df, window_df, dots_df = (pd.concat(frames[k]) for k in ("acf", "band", "window", "dots"))
for frame in (acf_df, band_df, window_df, dots_df):
    frame["series"] = frame["series"].map(TAGS)
label_df = dots_df.assign(y=dots_df["y"] + 0.12)

show(
    ggplot()
    + geom_band(aes(xmin="x0", xmax="x1"), data=window_df, fill=tint(BLUE, 0.88), size=0)
    + geom_ribbon(aes("lag", ymin="lo", ymax="hi"), data=band_df, fill=tint(SLATE, 0.6), size=0)
    + geom_line(aes("lag", "acf"), data=acf_df, color=SLATE, size=0.6)
    + geom_point(aes("x", "y"), data=dots_df, shape=21, fill="white", color=BLUE, size=3.2, stroke=1.2)
    + note(label_df.rename(columns={"x": "x"}), colour=BLUE, hjust=0.5, size=4.2)
    + facet_wrap("series", ncol=2, scales="fixed")
    + scale_x_continuous(breaks=list(range(8)))
    + labs(x="Lag (days)", y="Autocorrelation")
    + house() + ggsize(720, 440)
)

The analysis gives the same answer for all four series: a window of one day, shown as the blue band of @fig-acf, and lag candidates of two, three and seven days, shown as the dots. The answer follows from the daily cycle that every series has, and the figure gives the autocorrelation behind it. The charging of vehicles repeats most strongly, with an autocorrelation of 0.88 at one day, and it is $-0.55$ at half a day, where the load lies on the opposite side of the cycle. The PV array and the net load follow at 0.77 and 0.79. The household is the weakest at 0.35, since a single home does not repeat its day closely. Its autocorrelation at seven days, 0.42, exceeds the value at one day, which marks the week as the stronger rhythm of this household. Every series stays far above the band that marks the absence of autocorrelation, which is $\pm 0.011$ to $\pm 0.035$.

Two limits of this step should be stated. The candidate lags are calendar multiples of the detected period, so the analysis names the same candidates for any series with a daily cycle. It does not search for a lag that is specific to a series; it proposes where to look. The choice among the candidates, and between them and the other inputs, is made by the ranking of the next section.

## Measures of Relevance and Their Fusion

A candidate input is relevant to the extent that it tells something about the target. No single measure sees every kind of dependence, so twiga scores each candidate by six, which differ in the dependence that they can detect:

- the absolute Spearman rank correlation, for a monotone relation;
- the F statistic of a linear fit, for a linear relation;
- the mutual information, for any relation that the joint histogram of the two variables shows;
- Chatterjee's $\xi$ coefficient, a rank-based measure that is zero for independence and one for a functional relation of any shape;[^xi]
- the predictive power score, the gain of a small decision tree fitted on the candidate alone over predicting the mean of the target;
- the impurity importance of a random forest, which also sees interactions between inputs.[^forest]

A relation that rises and falls, such as a load that grows when the temperature departs from a comfortable value in either direction, is invisible to the rank correlation and the F statistic and visible to the others. A ranking built on one measure therefore inherits that measure's blind spot.

The scores lie on different scales, so they are converted to ranks and the ranks are combined, a procedure known as rank aggregation.[^dwork] Let $r_{ij}$ be the rank of candidate $i$ under measure $j$, with the best of $m$ candidates given the rank $m$. The Borda count sums the ranks,

$$
B_i = \sum_{j} r_{ij}.
$$

Several measures repeat each other, since the rank correlation and the F statistic nearly always order candidates alike, and a sum with equal weights counts such a view twice. twiga also offers a weighted sum, $S_i = \sum_j w_j\, r_{ij}$, whose weights follow the uniqueness of each measure. Weight $w_j$ is proportional to $1 - \bar{\rho}_j$, where $\bar{\rho}_j$ is the mean absolute Spearman correlation of the scores of measure $j$ with those of the others. The weights are floored at 0.05 and normalised to sum to one. The default of the library is the equal-weight sum, and the figures below use the weighted sum, the option `weighted_borda`.

Two further choices follow the practice of the book chapter. Each cyclic calendar variable enters as a sine and a cosine, so that 23:00 lies next to 00:00.[^fpp] A column of standard normal noise joins the candidates as a control, and a measure that ranks it above a real input overstates relevance.

[^xi]: S. Chatterjee, A new coefficient of correlation, *Journal of the American Statistical Association* 116(536), 2021, pp. 2009 to 2022.

[^forest]: L. Breiman, Random forests, *Machine Learning* 45(1), 2001, pp. 5 to 32.

[^dwork]: C. Dwork, R. Kumar, M. Naor and D. Sivakumar, Rank aggregation methods for the Web, *Proceedings of the 10th International Conference on World Wide Web*, 2001, pp. 613 to 622.

[^fpp]: R. J. Hyndman and G. Athanasopoulos, *Forecasting: Principles and Practice*, 3rd ed., OTexts, 2021, <https://otexts.com/fpp3/>.

In [ ]:
RANKERS = ["abs_corr", "f_score", "mi_score", "xi_score", "pps_score", "rf_importance"]
SHORT = {"abs_corr": "Rank correlation", "f_score": "F statistic", "mi_score": "Mutual information", "xi_score": "Chatterjee xi", "pps_score": "Predictive power", "rf_importance": "Forest importance"}
NOUN = {"Household demand": "Demand", "EV charging": "Charging", "PV output": "PV output", "Net load": "Net load"}
FIELDS = {
    "hour_sin": "Hour, sine", "hour_cos": "Hour, cosine", "wday_sin": "Weekday, sine", "wday_cos": "Weekday, cosine",
    "yday_sin": "Day of year, sine", "yday_cos": "Day of year, cosine", "noise": "Noise (control)",
    "solar_wm2": "Solar radiation", "air_temp_c": "Air temperature", "irradiance_wm2": "Irradiance", "panel_temp_c": "Panel temperature",
}


def label(feature: str, name: str) -> str:
    """Sentence-case label of a candidate column."""
    days = lambda steps: f"{int(steps) // SERIES[name]['samples']} days"  # noqa: E731
    if "_lag_" in feature:
        return f"{NOUN[name]} {days(feature.rsplit('_', 1)[1])} earlier"
    if "_rolling_mean_win_" in feature:
        return f"Mean of the last {days(feature.rsplit('_', 1)[1])}"
    return FIELDS.get(feature, feature)


def design(name: str, share: float = TRAIN_SHARE) -> tuple[pd.DataFrame, dict[str, list[str]], str]:
    """Candidate inputs of a series on its training window, in three subgroups, with a noise column as control."""
    series = SERIES[name]
    n, target = series["samples"], series["target"]
    _, days = lookback_and_lags(name, share)
    frame = AutoregressTransformer(
        n_samples=n, lags=days, windows=[days[0], days[-1]], window_funcs=["mean"], value_column=target
    ).fit_transform(training(name, share))
    frame = add_temporal_feature(frame, "timestamp")
    frame = add_fourier_features(frame, ["hour", "wday", "yday"], [24, 7, 366])
    frame["noise"] = np.random.default_rng(0).standard_normal(len(frame))
    groups = {
        "Lags and means": [c for c in frame.columns if "_lag_" in c or "_rolling_" in c],
        "Calendar": [f"{field}_{kind}" for field in ("hour", "wday", "yday") for kind in ("sin", "cos")],
        "Known inputs": series["known"],
    }
    return frame.dropna(subset=[c for g in groups.values() for c in g] + ["noise", target]), groups, target


def rank(frame: pd.DataFrame, features: list[str], target: str, aggregation: str = "weighted_borda") -> tuple[pd.DataFrame, np.ndarray]:
    """twiga's six measures and their fusion for the given candidates; also the diversity weights of the measures."""
    _, scores = select_top_features(
        frame, features, target, top_k=len(features), alpha=None, include_chi2=False,
        rank_aggregation=aggregation, return_scores=True,
    )
    scores = scores.set_index("feature")
    return scores, diversity_weights(scores[RANKERS].fillna(0))


def pooled(name: str, share: float = TRAIN_SHARE) -> tuple[pd.DataFrame, np.ndarray]:
    """One ranking of all the subgroups together and the noise column."""
    frame, groups, target = design(name, share)
    return rank(frame, [c for g in groups.values() for c in g] + ["noise"], target)


results = {name: pooled(name) for name in SERIES}
for name, (scores, weights) in results.items():
    print(f"{name}: {len(scores)} candidates; weights " + ", ".join(f"{SHORT[m].lower()} {w:.2f}" for m, w in zip(RANKERS, weights)))

In [ ]:
#| label: fig-agreement
#| fig-cap: "Rank correlation between the six measures on the net load, with the weight of each measure at the right. The darker the cell, the more two measures order the inputs alike."
scores, weights = results["Net load"]
SHORTNAME = {"abs_corr": "Rank corr.", "f_score": "F statistic", "mi_score": "Mutual info.", "xi_score": "Chatterjee xi", "pps_score": "Pred. power", "rf_importance": "Forest"}
corr = scores[RANKERS].fillna(0).corr(method="spearman")
order = [SHORTNAME[m] for m in RANKERS]
cells = pd.DataFrame(
    [(SHORTNAME[a], SHORTNAME[b], corr.loc[a, b]) for a in RANKERS for b in RANKERS], columns=["row", "column", "value"]
)
cells["label"] = cells["value"].map("{:.2f}".format)
cells["ink"] = np.where(cells["value"] > 0.88, "white", INK)
weights_df = pd.DataFrame({"row": order, "column": "Weight", "label": [f"{w:.2f}" for w in weights]})
print("Weights:", {m: round(float(w), 2) for m, w in zip(order, weights)}, "| smallest off-diagonal correlation:", round(corr.to_numpy()[np.triu_indices(6, 1)].min(), 2))

show(
    ggplot(cells, aes("column", "row"))
    + geom_tile(aes(fill="value"), color="white", size=1.0)
    + scale_fill_gradient(low="white", high=BLUE, limits=[0.6, 1.0])
    + geom_text(aes(label="label", color="ink"), size=5.0, family=FONT)
    + scale_color_manual(values={"white": "white", INK: INK})
    + geom_text(aes("column", "row", label="label"), data=weights_df, size=5.0, family=FONT, color=INK, fontface="bold", inherit_aes=False)
    + scale_x_discrete(limits=order + ["Weight"], position="top")
    + scale_y_discrete(limits=order[::-1])
    + labs(x="", y="")
    + house() + theme(axis_line=element_blank(), axis_ticks=element_blank(), axis_text_x=element_text(angle=0)) + ggsize(720, 380)
)

@fig-agreement shows how far the six measures agree on the net load. The most similar pair, the mutual information and $\xi$, agree at 0.98, and the least similar, the predictive power and the forest importance, at 0.65. The weights therefore stay between 0.14 and 0.19 for the net load, and between 0.13 and 0.25 across the four series, where the largest is that of the forest importance on the PV output. Because the measures repeat each other only in part, the diversity weights differ little from equal weights, and a later section checks how little.

## Ranking within Subgroups

The candidates form three subgroups with a different meaning: the lags and rolling means of the target, the calendar variables, and the known inputs such as the weather. Each subgroup is ranked on its own first, by the six measures and the equal-weight Borda count, which is the default of twiga. The noise column joins every subgroup as a control. A within-subgroup ranking answers questions that belong together, such as which lag is the best and which calendar encoding is the best, and it is the way that twiga's dataset loader selects inputs.

@fig-groups shows the three rankings for the net load. Each tile gives the rank that one measure assigns, with 1 the best and the darkest blue, and the last column gives the sum of the ranks.

In [ ]:
#| label: fig-groups
#| fig-cap: "Ranking of the net load inputs within three subgroups, with the noise column as a control. Each tile is the rank under one measure (Corr for the rank correlation, F, MI for the mutual information, Xi, PPS for the predictive power, RF for the forest importance), 1 is best and darkest, and the last column is the sum of the ranks."
def short(feature: str) -> str:
    """Compact row label for the subgroup panels: 2-day lag, 7-day mean, and the calendar and input labels."""
    days = lambda steps: int(steps) // SERIES["Net load"]["samples"]  # noqa: E731
    if "_lag_" in feature:
        return f"{days(feature.rsplit('_', 1)[1])}-day lag"
    if "_rolling_mean_win_" in feature:
        return f"{days(feature.rsplit('_', 1)[1])}-day mean"
    return FIELDS.get(feature, feature)


def group_panel(frame: pd.DataFrame, group: str, features: list[str], target: str, tag: str):
    """Rank within one subgroup, with the noise column as its control: a heat panel of the rank under each measure."""
    scores, _ = rank(frame, features + ["noise"], target, aggregation="borda_count")
    ranks = scores[RANKERS].fillna(0).rank(ascending=False, method="first").astype(int)
    order = scores["borda_count"].sort_values(ascending=False).index.tolist()
    m = len(order)
    rows = [
        (short(f), {"abs_corr": "Corr", "f_score": "F", "mi_score": "MI", "xi_score": "Xi", "pps_score": "PPS", "rf_importance": "RF"}[r], int(ranks.loc[f, r]), 1 - (ranks.loc[f, r] - 1) / (m - 1))
        for f in order for r in RANKERS
    ]
    cells = pd.DataFrame(rows, columns=["row", "column", "rank", "shade"])
    cells["ink"] = np.where(cells["shade"] > 0.55, "white", INK)
    columns = list(dict.fromkeys(cells["column"]))
    fused = pd.DataFrame({"row": [short(f) for f in order], "column": "Sum", "label": [f"{scores.loc[f, 'borda_count']:.0f}" for f in order]})
    names = [short(f) for f in order]
    plot = (
        ggplot(cells, aes("column", "row"))
        + geom_tile(aes(fill="shade"), color="white", size=1.0)
        + scale_fill_gradient(low="white", high=BLUE, limits=[0, 1])
        + geom_text(aes(label="rank", color="ink"), size=4.6, family=FONT)
        + scale_color_manual(values={"white": "white", INK: INK})
        + geom_text(aes("column", "row", label="label"), data=fused, size=4.6, family=FONT, color=INK, fontface="bold", inherit_aes=False)
        + scale_x_discrete(limits=columns + ["Sum"], position="top", expand=[0.0, 0.8])
        + scale_y_discrete(limits=names[::-1])
        + labs(title=f"({tag}) {group}", x="", y="")
        + house()
        + theme(axis_line=element_blank(), axis_ticks=element_blank(), plot_title=element_text(face="bold", size=12, hjust=0), axis_text_x=element_text(size=9.5, angle=0))
    )
    return plot, scores


frame, groups, target = design("Net load")
panels = []
for tag, (group, features) in zip("abc", groups.items()):
    plot, scores = group_panel(frame, group, features, target, tag)
    panels.append(plot)
    best = [label(f, "Net load") for f in scores["borda_count"].sort_values(ascending=False).index]
    print(f"{group}: order {best}")

show(gggrid(panels, ncol=3) + ggsize(1080, 340))

Among the lags and means of the net load, the lag of two days is first, followed by the lags of seven and three days and then the two means of the past, and the noise column is last. Among the calendar variables the sine of the hour comes first, the cosine of the day of the year second and the cosine of the hour third. Both weekday encodings rank below the day of the year, and the noise column is again last. Among the known inputs the solar radiation is first, the air temperature second and the noise third, so both weather columns carry more information than the control.

A rule that keeps the best two lags or means, the best two calendar variables and the best known input from these rankings gives a fixed quota per subgroup. The printed comparison below sets that rule against the ranking of all candidates together, which the next section describes. For the household and for the EV charging the quota rule and the combined ranking select the same four inputs. For the PV output and the net load they share three of five. The quota keeps two calendar variables that rank seventh to ninth of fourteen when the candidates compete, and it leaves out the lag of three days of the net load, which ranks third, and the panel temperature of the PV array, which ranks fourth. None of the kept inputs falls below the noise column in the combined order.

In [ ]:
def per_group_rule(name: str) -> list[str]:
    """The best two lags or means, the best two calendar inputs and the best known input, each ranked within its subgroup."""
    frame, groups, target = design(name)
    picks = []
    for group, features in groups.items():
        if not features:
            continue
        scores, _ = rank(frame, features + ["noise"], target, aggregation="borda_count")
        ordered = [f for f in scores["borda_count"].sort_values(ascending=False).index if f != "noise"]
        picks += ordered[: 1 if group == "Known inputs" else 2]
    return picks


for name, (scores, _) in results.items():
    pooled_order = scores["weighted_borda"].sort_values(ascending=False).index.tolist()
    picks = per_group_rule(name)
    position = {f: pooled_order.index(f) + 1 for f in picks}
    below_noise = [label(f, name) for f in picks if position[f] > pooled_order.index("noise") + 1]
    print(f"{name}: per-subgroup picks {[label(f, name) for f in picks]}; positions in the pooled order {list(position.values())}; shared with the pooled top {len(picks)}: {len(set(picks) & set(pooled_order[: len(picks)]))}; below the noise column: {below_noise or 'none'}")

## One Ranking across the Subgroups

The same candidates are then ranked together, so that a lag competes with a calendar variable and with a weather column on one scale. The ranking reads the first 70% of each series and uses the diversity-weighted sum, the option `weighted_borda`. There are twelve candidates for the household and the charging, and fourteen for the PV output and the net load.

@fig-gap draws the fused score of every candidate, best at the top, with the five best in blue and the noise column in orange. The length of five is a choice made for the figure. The dashed line marks the cut after five, and the label gives the fall in score after it as a share of the range from the best to the worst.

In [ ]:
#| label: fig-gap
#| fig-cap: "Fused score of every input, best at the top. The dashed line marks the cut after five inputs, blue dots are the shortlist, and the orange dot is the noise control."
frames, cuts = [], []
for name, (scores, _) in results.items():
    fused = scores["weighted_borda"].sort_values(ascending=False)
    n = len(fused)
    frames.append(
        pd.DataFrame(
            {
                "series": name,
                "position": np.arange(n, 0, -1),
                "score": fused.to_numpy(),
                "label": [label(f, name) for f in fused.index],
                "group": np.where(fused.index == "noise", "Noise", np.where(np.arange(n) < 5, "Shortlist", "Other")),
            }
        )
    )
    share = (fused.iloc[4] - fused.iloc[5]) / (fused.iloc[0] - fused.iloc[-1])
    cuts.append(pd.DataFrame({"series": [name], "x": [fused.iloc[0]], "y": [n - 4.5], "label": [f"Gap {share:.1%} of range"]}))
    print(f"{name}: fifth {fused.iloc[4]:.2f}, sixth {fused.iloc[5]:.2f}, gap {share:.1%} of the range; noise at position {int(fused.index.get_loc('noise')) + 1} of {n}")
gaps, cut_df = pd.concat(frames), pd.concat(cuts)
gaps["zero"] = 0.0
labels_df = gaps.assign(x=-0.3, y=gaps["position"])
for frame in (gaps, cut_df, labels_df):
    frame["series"] = frame["series"].map(TAGS)

show(
    ggplot(gaps)
    + geom_segment(aes(x="zero", xend="score", y="position", yend="position", color="group"), size=1.0)
    + geom_point(aes("score", "position", color="group", fill="group"), shape=21, size=3.4, stroke=1.1)
    + geom_segment(aes(x="left", xend="right", y="y", yend="y"), data=cut_df.assign(left=-11.0, right=17.0), color=INK, linetype="dashed", size=0.4, inherit_aes=False)
    + scale_color_manual(values={"Shortlist": BLUE, "Other": GREY_DARK, "Noise": ORANGE})
    + scale_fill_manual(values={"Shortlist": BLUE, "Other": "white", "Noise": ORANGE})
    + note(labels_df, colour=INK, hjust=1.0, size=4.2)
    + note(cut_df.assign(x=11.0, y=cut_df["y"] - 0.9), colour=INK, hjust=0.0, size=4.2)
    + facet_wrap("series", ncol=2)
    + scale_x_continuous(limits=[-11.5, 17], breaks=[0, 5, 10, 15])
    + labs(x="Fused score (higher is more relevant)", y="")
    + house() + theme(axis_line_y=element_blank(), axis_ticks_y=element_blank(), axis_text_y=element_blank()) + ggsize(900, 660)
)

In [ ]:
for name, (scores, _) in results.items():
    n = len(scores)
    by_measure = {SHORT[m].lower(): int(scores[m].fillna(0).rank(ascending=False, method="first")["noise"]) for m in RANKERS}
    fused = int(scores["weighted_borda"].rank(ascending=False, method="first")["noise"])
    print(f"{name}: position of the noise column among {n}, by measure {by_measure}; fused {fused}")

The four orders give a consistent picture, with differences that follow the physics of each series.

For the **household**, the lag of seven days comes first, the sine and the cosine of the hour second and third, and the lags of two and three days fourth and fifth. The two weekday encodings rank tenth and eleventh. The household therefore repeats by the week through its own past and not through a weekday variable. The noise column is last of twelve under every measure except the forest importance, which places it eighth.

For **EV charging**, the lag of seven days is first, the lag of two days second, the sine of the hour third, the cosine of the hour fourth and the lag of three days fifth. The weekday encodings and the noise are the last three.

For **PV output**, the irradiance is first, the lags of two and three days follow, the panel temperature is fourth and the lag of seven days fifth. The noise column is twelfth of fourteen, above the two weekday encodings, which fits an array whose output does not depend on the weekday.

For the **net load**, the lags of two, seven and three days come first, then the solar radiation. The air temperature is ninth, below two means of the past and two calendar variables, and the noise column is last of fourteen.

Across the four series the noise is last or among the three lowest. The printed positions show that one measure, the forest importance, ranks the noise column above real inputs in the household, the PV output and the net load. The fusion repairs that, which is the practical case for combining measures. The shortlist of five is not equally firm: the fall in score after the fifth input is 12.7% of the range for the EV charging, a break that supports the cut, and 2.2%, 3.5% and 4.8% for the household, the PV output and the net load, where the sixth input is nearly as relevant as the fifth. An input that scores near the noise column, as the weekday encodings do, can be removed without loss. The ranking also keeps the sine and the cosine of the hour side by side in two of the four shortlists, although a model needs the pair only to place the hour on a circle, so relevance alone does not remove redundancy.

## Equal Weights and Diversity Weights

@fig-compare sets the order of twiga's default, the equal-weight Borda count, against the diversity-weighted order. In all four series the two orders select the same five inputs, and their rank correlation lies between 0.99 and 1.00. The weighting does not change the shortlist here, so equal weights are a sound default and the diversity weights a refinement whose benefit is small on this evidence.

In [ ]:
#| label: fig-compare
#| fig-cap: "Position of each input under equal weights and under diversity weights. Points on the dotted line keep their position, blue points are the shortlist, and the orange point is the noise control."
pairs = []
for name, (scores, _) in results.items():
    equal = scores["borda_count"].rank(ascending=False, method="first")
    weighted = scores["weighted_borda"].rank(ascending=False, method="first")
    group = np.where(scores.index == "noise", "Noise", np.where(weighted <= 5, "Shortlist", "Other"))
    pairs.append(pd.DataFrame({"series": name, "equal": equal.to_numpy(), "weighted": weighted.to_numpy(), "group": group}))
pairs = pd.concat(pairs)
pairs["series"] = pairs["series"].map(TAGS)
def agreement_label(scores: pd.DataFrame) -> str:
    same = set(scores["borda_count"].nlargest(5).index) == set(scores["weighted_borda"].nlargest(5).index)
    return f"Same five inputs: {'yes' if same else 'no'}; rank correlation {scores['borda_count'].corr(scores['weighted_borda'], method='spearman'):.3f}"


agree = pd.DataFrame(
    {"series": [TAGS[name] for name in results], "x": 1.0, "y": [len(r[0]) + 0.6 for r in results.values()], "label": [agreement_label(r[0]) for r in results.values()]}
)
print(agree["label"].to_string(index=False))

show(
    ggplot(pairs, aes("equal", "weighted"))
    + geom_abline(slope=1, intercept=0, color=GREY_DARK, linetype="dotted")
    + geom_point(aes(color="group", fill="group"), shape=21, size=4, stroke=1.1)
    + scale_color_manual(values={"Shortlist": BLUE, "Other": GREY_DARK, "Noise": ORANGE})
    + scale_fill_manual(values={"Shortlist": BLUE, "Other": "white", "Noise": ORANGE})
    + note(agree, colour=INK, hjust=0.0, size=4.2)
    + facet_wrap("series", ncol=2)
    + scale_x_continuous(breaks=[1, 5, 10, 14])
    + scale_y_continuous(breaks=[1, 5, 10, 14])
    + labs(x="Position under equal weights (1 is best)", y="Position under diversity weights")
    + house() + ggsize(720, 560)
)

## Stability across Training Windows

A ranking is of little use if it changes when a few more weeks of data arrive. @fig-stability repeats the whole analysis on the first 50%, 70% and 90% of each series, with the lookback and the lags found again each time. The top input is the same in all three windows for every series, and the top five inputs are identical, so the mean overlap is 1.00. The windows are nested, however, so each shares data with the others. The test shows stability as data accumulate and not independence across periods. A check on a later, disjoint period would be stronger, and a longer record would allow it. The noise control is last in the household and the charging at every window. It is within the two lowest for the net load at every window, and within the three lowest for the PV output, where two weekday encodings fall below it at 70% and 90%.

In [ ]:
#| label: fig-stability
#| fig-cap: "Position of each input in the fused order when the analysis uses the first 50%, 70% and 90% of the series. Blue lines are the shortlist at 70%, orange is the noise control, grey are the other inputs."
WINDOWS = (0.5, 0.7, 0.9)
orders = {}
for name in SERIES:
    for share in WINDOWS:
        scores, _ = results[name] if share == TRAIN_SHARE else pooled(name, share)
        orders[(name, share)] = scores["weighted_borda"].sort_values(ascending=False).index.tolist()

lines, ends = [], []
for name in SERIES:
    keep = orders[(name, TRAIN_SHARE)][:5]
    for share in WINDOWS:
        for position, feature in enumerate(orders[(name, share)], start=1):
            group = "Noise" if feature == "noise" else "Shortlist" if feature in keep else "Other"
            lines.append((name, feature, int(share * 100), position, group))
    for feature in keep + ["noise"]:
        ends.append((name, 92.0, orders[(name, WINDOWS[-1])].index(feature) + 1, label(feature, name)))
rank_df = pd.DataFrame(lines, columns=["series", "feature", "window", "rank", "group"])
end_df = pd.DataFrame(ends, columns=["series", "x", "y", "label"])
for frame in (rank_df, end_df):
    frame["series"] = frame["series"].map(TAGS)
for name in SERIES:
    top5 = [set(orders[(name, s)][:5]) for s in WINDOWS]
    noise = [orders[(name, s)].index("noise") + 1 for s in WINDOWS]
    print(f"{name}: top five identical in all three windows: {top5[0] == top5[1] == top5[2]}; noise position {noise} of {len(orders[(name, 0.7)])}")

show(
    ggplot(rank_df, aes("window", "rank"))
    + geom_line(aes(group="feature", color="group", size="group"))
    + geom_point(aes(color="group", fill="group"), shape=21, size=2.6, stroke=1.0)
    + scale_color_manual(values={"Shortlist": BLUE, "Other": GREY, "Noise": ORANGE})
    + scale_fill_manual(values={"Shortlist": BLUE, "Other": "white", "Noise": ORANGE})
    + scale_size_manual(values={"Shortlist": 1.0, "Other": 0.5, "Noise": 1.0})
    + note(end_df, colour=INK, hjust=0.0, size=4.2)
    + facet_wrap("series", ncol=2)
    + scale_x_continuous(breaks=[50, 70, 90], labels=["50%", "70%", "90%"], limits=[46, 122])
    + scale_y_reverse(breaks=[1, 5, 10, 14])
    + labs(x="Share of the series used for training", y="Position in the fused order")
    + house() + ggsize(900, 640)
)

## Limits of the Method

Several limits bear on how far the ranking can be trusted.

- **Relevance is not usefulness.** Each measure scores one input at a time. Two inputs that carry the same information both rank high, so a shortlist can hold redundant inputs. The ranking is a shortlist for a forecaster to test and not a final choice.
- **Accuracy is not tested.** The post does not fit a forecaster, so it does not show that the shortlist forecasts better than the full set of candidates. A comparison on held-out periods is the natural next step, and its result may differ by series.
- **Known inputs are assumed.** The weather columns are treated as known over the forecast period. With a real weather forecast their relevance would be lower.
- **The candidates and the cut are fixed.** The lags are calendar multiples of the detected period, and other lags, such as half-day multiples, are not in the pool. The shortlist length of five is a convention.
- **Selection reads the training window only.** twiga's `load_dataset` applies the same functions to the whole file, which suits the choice of a configuration for a dataset. A backtest has to call the functions on the training window, as the code above does, or the ranking sees the period that it is later tested on.
- **One household, one site, one city.** The findings describe these series and do not show how common they are.

## How to Apply This

The steps below follow the order of the post, and every step uses the training window only.

1. Hold out the period to be forecast, and take the first part of the series in time.
2. Find the periods of the series from its autocorrelation. The shortest period sets the lookback window, and the longer periods become lag candidates, as `get_best_autoregessive_features` returns them.
3. List the candidates in subgroups: the lags and rolling means, the sine and cosine of each calendar cycle, and the inputs that are known in advance. Add a column of noise.
4. Rank each subgroup, then all candidates together. Keep the inputs that rank above the noise column and above a visible gap in the scores, and drop the rest.
5. Treat the shortlist as the input to a forecaster, and test it against the full set of candidates on held-out periods.

## Summary

On all four series the autocorrelation gives a window of one day and lag candidates of two, three and seven days, which follows from their daily cycles. The ranking then gives these results.

- The lag of seven days is the best input for the household and the EV charging, the lag of two days for the net load, and the irradiance for the PV output.
- The hour is the only calendar variable that enters a shortlist, and the weekday encodings rank among the lowest in every series.
- The noise control ranks last or among the lowest three in all four series, while the forest importance alone ranks it above real inputs in three.
- Ranking within subgroups and then together selects the same inputs for the household and the EV charging, and a fixed quota per subgroup keeps weaker calendar inputs for the PV output and the net load.
- Equal weights and diversity weights select the same five inputs in every series.
- The top five inputs do not change between the first 50%, 70% and 90% of a series, though these windows are nested.

The ranking is a shortlist to test and not a verdict, and the size of the shortlist should be read from the gaps between scores. The notebook and the two small data files are in [the repository of this site](https://github.com/sambaiga/sambaiga.github.io/tree/main/pages/blog/posts/2026/11), and the EV series is downloaded when the notebook runs. Run it with `twiga[ml]==1.1.8` and lets-plot. The method, its measures and its checks on synthetic series with a known answer are developed in Chapter 2 of *Forecasting for Operations*.